In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
import matplotlib.pyplot as plt

#https://www.kaggle.com/datasets/davidcariboo/player-scores
df = pd.read_csv("../players.csv")

df['foot_encoded'] = df['foot'].map({'left': 0, 'right': 1})
data = df[['height_in_cm', 'market_value_in_eur', 'foot_encoded']].dropna()

# log transform market value
data['market_value_in_eur'] = np.log1p(data['market_value_in_eur'])

X = data[['height_in_cm', 'market_value_in_eur']].values
y = data['foot_encoded'].values

# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# z-score
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

print(f"Training set size: {X_train_poly.shape[0]}")
print(f"Test set size: {X_test_poly.shape[0]}")

Training set size: 21926
Test set size: 5482


In [10]:
class LogisticRegression:
    def __init__(self, learning_rate=0.01, epochs=1000):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = None

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        num_samples, num_features = X.shape
        self.weights = np.zeros(num_features)
        self.bias = 0

        for _ in range(self.epochs):
            linear_model = np.dot(X, self.weights) + self.bias
            y_predicted = self.sigmoid(linear_model)

            dw = (1 / num_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / num_samples) * np.sum(y_predicted - y)

            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

    def predict(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        y_predicted = self.sigmoid(linear_model)
        return [1 if i > 0.5 else 0 for i in y_predicted]

# training..
lr = LogisticRegression(learning_rate=0.01, epochs=1000)
lr.fit(X_train_poly, y_train)

# predicting with our model
y_pred = lr.predict(X_test_poly)

print("IMpl logistic regresion accuracy:", accuracy_score(y_test, y_pred))

IMpl logistic regresion accuracy: 0.7298431229478293


In [ ]:
# compare with scikit-learn
lr_sklearn = LogisticRegression()
lr_sklearn.fit(X_train, y_train)
y_pred_sklearn = lr_sklearn.predict(X_test)

print("scikit-learn logistic reg accuracy:", accuracy_score(y_test, y_pred_sklearn))

print("our impl report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("scikit-learn report:")
print(classification_report(y_test, y_pred_sklearn, zero_division=0))

# check for similarity
similar_predictions = np.sum(np.array(y_pred) == y_pred_sklearn)
total_predictions = len(y_test)
print(f"match between our impl and scikit: {similar_predictions}/{total_predictions} ({similar_predictions/total_predictions*100:.2f}%)")